# Hypothesis Testing

**If you are a Colab user**

If you use Google Colab, uncomment the following cell to mount your Google Drive in Colab. <br>
After that, Colab can read and write files and data in your Google Drive. <br>

Please change the current directory to the folder where you saved your notebook and the data folder. For example, I save my Colab files and data in the following location.

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

#%cd /content/drive/MyDrive/Colab\ Notebooks

**Set up standards for the remainder of the notebook**

In [2]:
# import libraries and modules used in this notebook
import numpy as np 
np.set_printoptions(precision=4, suppress=True)
np.random.seed(12345)

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import scipy as sp
from scipy import stats # we use the stats module from scipy

import statsmodels as sm
from statsmodels.stats import proportion # we use it for z tests

# display multiple outputs in one cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Import and Review California Housing Data

The `CA_housing.csv` dataset contains housing information for California districts, including summary statistics based on the 1990 census. The dataset contains 20,640 observations and 10 columns.

Below is a list of the 9 attributes (X) with their descriptions.

  --Longitude: block group longitude\
  --Latitude: block group latitude\
  --HouseAge: median house age in the block group\
  --AveRooms: average number of rooms per household\
  --AveBedrms: average number of bedrooms per household\
  --Population: block group population\
  --AveOccup: average number of household members\
  --MedInc: median household income within a block\
  --OceanProx: location of the house with respect to the ocean/sea

The target (y) is:\
--MedVal: median house value for households within a block


In [3]:
# read the housing dataset and become familiar with the data types
CA_housing=pd.read_csv("Data/CA_housing.csv")
CA_housing.head()
CA_housing.info()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,OceanProx,MedVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,NEAR BAY,452600.0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,NEAR BAY,358500.0
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,NEAR BAY,352100.0
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,NEAR BAY,341300.0
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,NEAR BAY,342200.0


<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
 8   OceanProx   20640 non-null  str    
 9   MedVal      20640 non-null  float64
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [4]:
# change 'OceanProx' to an ordered categorical variable
ordered_categories = ['INLAND', 'NEAR BAY', '<1H OCEAN', 'NEAR OCEAN', 'ISLAND']
CA_housing['OceanProx'] = pd.Categorical(CA_housing['OceanProx'], 
                                         categories=ordered_categories, 
                                         ordered=True
                                        )

## Hypothesis Testing Foundation

**Hypothesis**

A hypothesis is a statement about a population quantity, such as a mean, proportion, or distribution. In hypothesis testing, we compare two competing statements: the *null hypothesis* ($H_0$), which usually represents the baseline claim, and the *alternative hypothesis* ($H_a$), which represents the claim we are looking for evidence to support.

Sample data do not prove that one hypothesis is absolutely true. Instead, they help us decide whether the observed sample result would be unusual if the null hypothesis were true.

**Steps of Hypothesis Testing**
1. Develop the null and alternative hypotheses.
2. Specify the significance level, $\alpha$.
3. Collect the sample data and compute the value of the test statistic.
4. Use the value of the test statistic to compute the $p$-value.
5. Make a decision about the null hypothesis.
6. Interpret the statistical conclusion in the context of the application.

## One-sample t test

The one-sample t test is often used to test a hypothesis about $\mu$, the mean of a distribution $F$. Let $\mu_0$ be the hypothesized value of the mean, and let $\alpha$ be significance level in hypothesis testing. Suppose we draw a random sample, $x_1,\dots,x_n$, from the distribution $F$. We use this sample to test whether the population mean is consistent with the hypothesized value.

The table below summarizes three types of one-sample t tests. The choice among lower-tail, upper-tail, and two-tail tests depends on the direction of the research question. 

**Table 1: Hypotheses for one-sample t tests**
$$
\begin{array}{l|c|c|c}
 & \text{lower-tail test} & \text{upper-tail test} & \text{two-tail test} \\
\hline
\text{Null hypothesis } H_0 & \mu \geq \mu_0 & \mu \leq \mu_0 & \mu = \mu_0 \\
\text{Alternative hypothesis } H_1 & \mu < \mu_0 & \mu > \mu_0 & \mu \neq \mu_0 \\
p\text{-value} & P(X \leq t) & P(X > t) & P(X \geq |t|) \\
\hline
\end{array}
$$

The test statistic, t, is
\begin{equation}
t=\frac{\bar{x}-\mu_0}{s/\sqrt{n}}. \tag{1}
\end{equation}

This statistic measures how far the sample mean is from the hypothesized mean, using the standard error as the unit of comparison. The $p$-value represents the probability of observing a test statistic at least as extreme as the one calculated from the sample, assuming the null hypothesis is true. If the $p$-value is less than $\alpha$, the sample result is unusual enough that we reject the null hypothesis.

Let us test a hypothesis about $\mu$, the mean value of log(MedVal).

If the hypothesized mean of `log(MedVal)` for a survey block is 14, can you generate a random sample of size 1,000 and use the sample to perform the lower-tailed test, upper-tailed test, and two-tailed test, respectively? 


H0: $\mu$ = 14\
Ha: $\mu$ != 14, < 14, or > 14, depending on the belief or conjecture that you have about this parameter

We use **ttest_1samp(sample, hypothesized value, alternative=)** in `scipy.stats` to perform a one-sample t test. Use `alternative = 'less'` for a lower-tail test, `'greater'` for an upper-tail test, and `'two-sided'` for a two-tail test. After each test, compare the $p$-value with `alpha` to decide whether to reject $H_0$.

In [5]:
# sp.stats.ttest_1samp(sample, hypothesized value, alternative) performs the one-sample t test

alpha =0.05 # significance level
n=1000 # sample size
mu_0 = 14 # hypothesized value of the mean

sample_ppl = np.log(CA_housing.MedVal.sample(n)) # a random sample of size n
print(f'The sample mean is {sample_ppl.mean():.2f}')
print(f'The sample standard deviation is {sample_ppl.std(ddof=1):.3f}','\n')

# lower-tail test (alternative: mu < mu_0)
t_stat, p_value = sp.stats.ttest_1samp(sample_ppl, mu_0, alternative='less')
print('lower-tail test:')
print(f'Test statistic is {t_stat:.3f} and the p value is {p_value:.3f}')
if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

# upper-tail test (alternative: mu > mu_0)
t_stat, p_value = sp.stats.ttest_1samp(sample_ppl, mu_0, alternative='greater')
print('upper-tail test:')
print(f'Test statistic is {t_stat:.3f} and the p value is {p_value:.3f}')
if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

# two-tail test (alternative hypothesis: mu != mu_0)
t_stat, p_value = sp.stats.ttest_1samp(sample_ppl, mu_0, alternative='two-sided')
print('two-tail test:')
print(f'Test statistic is {t_stat:.3f} and the p value is {p_value:.3f}')
if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')


The sample mean is 12.10
The sample standard deviation is 0.577 

lower-tail test:
Test statistic is -103.947 and the p value is 0.000
Reject the null hypothesis at the significance level 0.05 

upper-tail test:
Test statistic is -103.947 and the p value is 1.000
Fail to reject the null hypothesis at the significance level 0.05 

two-tail test:
Test statistic is -103.947 and the p value is 0.000
Reject the null hypothesis at the significance level 0.05 



## One-way ANOVA Test / F test

The one-way ANOVA test, also called the F test, tests whether two or more groups have the same mean. It is useful when we have more than two groups and want to avoid running many pairwise t tests as the first step.

Suppose there are $m$ distributions whose mean values are $\mu_i$, for $i=1,\dots,m$. The hypotheses of the one-way ANOVA test are:

H0: $\mu_1=\cdots\mu_m$

Ha: $\exists\; \mu_i\neq\mu_j$, for $i, j \in\{1,\dots,m\}$

The null hypothesis says that all group means are equal. The alternative hypothesis says that at least one group mean is different, but it does not tell us which pair is different.

We often perform the one-way ANOVA test first. If we reject the null hypothesis, we can then perform two-sample t tests to further examine the relationship between the means of any pair of distributions.

Assume that a random sample is drawn from each of the $m$ distributions.
$n_j$, $\overline{x}_j$, and $s_j$ are the sample size, sample mean, and sample standard deviation for distribution $j$, for $j=1,\dots,m$. Let $\overline{x}$ be the overall mean:
\begin{equation}
\overline{x}=\frac{\sum_{j=1}^m \overline{x}_j n_j}{\sum_{j=1}^m n_j}. \tag{2}
\end{equation}
The test statistic $F$ is
\begin{equation}
F=\frac{\text{between-group mean squares}}{\text{Within-group mean squares}}=\frac{\sum_{j=1}^m (\overline{x}_j-\overline{x})^2 n_j\left/(m-1)\right.}{\sum_{j=1}^m s_j^2(n_j-1)\left/(\sum_{j=1}^m n_j-m)\right.} \tag{3}
\end{equation}
Intuitively, the F statistic compares variation between group means with variation within the groups. A large F statistic suggests that the group means are more spread out than we would expect from within-group variation alone. The corresponding $p$-value is the probability that a random value drawn from the F distribution with $(m-1)$ and $(\sum_{j=1}^m n_j-m)$ degrees of freedom is greater than the test statistic F.



Does the mean of log(MedVal) for houses in a block vary with the block's proximity to the ocean? If we split blocks by their proximity to the ocean, there are five groups. Can you generate a random sample of size 1,000 from each group and then use the samples to test whether the mean value of log(MedVal) in a block varies by proximity to the ocean? 

H0: $
\mu_\text{<1H OCEAN} = \mu_\text{NEAR BAY} = \mu_\text{INLAND} = \mu_\text{NEAR OCEAN}
$

Ha:
$
\mu_i \neq \mu_j \quad \text{for any } i, j \in \{\text{<1H OCEAN}, \text{NEAR BAY}, \text{NEAR OCEAN}, \text{INLAND}\}
$

Use **f_oneway(smp1, smp2,...)** in `scipy.stats`. If the ANOVA test rejects $H_0$, the result tells us that at least one group mean differs, but it does not identify the specific pair of groups responsible for the difference.

In [6]:
alpha =0.05 # significance level
n=1000 # sample size

# sample 1000 MedVal values from each group and apply the log function to the samples
samples = {
    i: np.log(CA_housing.loc[CA_housing.OceanProx == i, "MedVal"].sample(n))
    for i in ["<1H OCEAN", "NEAR BAY", "NEAR OCEAN", "INLAND"]
}


F_stat,p_value=sp.stats.f_oneway(samples["<1H OCEAN"],
                                 samples["NEAR BAY"],
                                 samples["NEAR OCEAN"],
                                 samples["INLAND"]
                                )

print('One-way ANOVA test:')
print(f'The test statistic is {F_stat:.3f} and the p value is {p_value:.3f}')

if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

One-way ANOVA test:
The test statistic is 524.685 and the p value is 0.000
Reject the null hypothesis at the significance level 0.05 



## Two-sample t test

The two-sample $t$ test is often used to test whether two distributions have the same mean value. 

Let $\mu_1$ and $\mu_2$ denote the means of two distributions, respectively. A random sample is drawn from each distribution. $n_i$, $\overline{x}_i$, and $s_i$ are the sample size, sample mean, and sample standard deviation of group $j$, for $j=1$ and $2$.


The table below summarizes the three types of two-sample t tests. Again, the direction of the alternative hypothesis should match the question we want to answer.

**Table 2: Hypotheses for two-sample t tests**
$$
\begin{array}{l l l l}
 & \text{lower-tailed test} & \text{upper-tailed test} & \text{two-tailed test} \\
\hline
\text{Null hypothesis } H_0: & \mu_1 \ge \mu_2 & \mu_1 \le \mu_2 & \mu_1 = \mu_2 \\
\text{Alternative hypothesis } H_a: & \mu_1 < \mu_2 & \mu_1 > \mu_2 & \mu_1 \neq \mu_2 \\
\text{p-value:} & P(x \le t) & P(x > t) & P(x \ge |t|) \\
\hline 
\end{array}
$$

**Unequal variance assumed**

If equal variances are not assumed, the test statistic for the two-sample t test is 
\begin{equation}
t=\frac{\overline{x}_1-\overline{x}_2}{\sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}}, \tag{4}
\end{equation}
and The $p$-value is calculated using a t distribution with degrees of freedom approximated by the Welch-Satterthwaite formula:
\begin{equation}
\mathrm{df}=\frac{\left(\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}\right)^2}{\frac{(s_1^2/n_1)^2}{n_1-1}+\frac{(s_2^2/n_2)^2}{n_2-1}}.\tag{5}
\end{equation}

**Equal variance assumed**

If equal variance of the two distributions is assumed, the test statistic  for the two-sample t test is
\begin{equation}
t=\frac{\overline{x}_1-\overline{x}_2}{\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}\sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}}}, \tag{6}
\end{equation}
and the $p$-value for each type of hypothesis test listed in the table is calculated using the t distribution with the degrees of freedom:
\begin{equation}
\mathrm{df} = n_1+n_2-2. \tag{7}
\end{equation}

In practice, the unequal-variance version is often safer when we are not sure whether the two populations have the same variance. In `scipy.stats.ttest_ind`, this choice is controlled by the `equal_var` argument.


What is the relationship between the mean value of log(MedVal), the log median value of houses in a block, for blocks near the ocean and those near the bay? You may draw a random sample of size 2,000 from each group and use the two independent samples to perform a hypothesis test that compares the mean values of MedVal for the two distributions.

H0: $\mu_{\text{NEAR BAY}}=\mu_{\text{NEAR OCEAN}}$

Ha:  $\mu_{\text{NEAR BAY}} \neq \mu_{\text{NEAR OCEAN}}$

We can use **ttest_ind(sample 1, sample 2, equal_var, alternative=types of test)** in `scipy.stats`. Because the alternative hypothesis uses $\neq$, this is a two-tailed test: we are asking whether the two means are different in either direction.

In [7]:
# two independent simple random samples of size 2000 (n=2000): one from "NEAR OCEAN" blocks and the other from "NEAR BAY" blocks


alpha =0.05 # significance level
n=2000 # sample size

# sample 2000 MedVal values from each group and apply the log function to the samples
samples = {
    i: np.log(CA_housing.loc[CA_housing.OceanProx == i, "MedVal"].sample(n))
    for i in ["NEAR BAY", "NEAR OCEAN"]
}

# test whether the two distributions have the same mean of MedVal using
# sp.stats.ttest_ind(sample 1, sample 2, alternative=types of test)
t_stat,p_value=sp.stats.ttest_ind(samples["NEAR BAY"],
                                  samples["NEAR OCEAN"],
                                  nan_policy='omit', 
                                  equal_var=False,
                                  alternative='two-sided') # alternative='two-sided', 'less', or 'greater'. Note: you need a recent version of scipy to use "alternative="
print('two-sample t test:')
print(f'The test statistic is {t_stat:.3f} and the p value is {p_value:.3f}')


if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

two-sample t test:
The test statistic is 2.718 and the p value is 0.007
Reject the null hypothesis at the significance level 0.05 



## One-sample z-test

The one-sample $z$ test is often used to test a hypothesis about $p$, the probability of success in a Bernoulli distribution. Let $p_0$ be the hypothesized probability of success, and let $\alpha$ be the level of significance in hypothesis testing. A random sample is selected with a sample proportion $\overline{p}$. The one-sample z test is summarized in the following table:

**Table 3: Hypotheses for one-sample z tests**

$$
\begin{array}{l l l l}
 & \text{lower-tail test} & \text{upper-tail test} & \text{two-tail test} \\
\hline
\text{Null hypothesis } H_0: & p \ge p_0 & p \le p_0 & p = p_0 \\
\text{Alternative hypothesis } H_a: & p < p_0 & p > p_0 & p \neq p_0 \\
\text{p-value:} & P(x \le z) & P(x > z) & P(x \ge |z|) \\
\hline
\end{array}
$$


The test statistic is
\begin{equation}
z=\frac{\overline{p}-p_0}{\sqrt{\frac{p_0(1-p_0)}{n}}}. \tag{8}
\end{equation}

The z statistic measures how far the sample proportion is from the hypothesized proportion, using the standard error as the unit of comparison. The $p$-value corresponding to the test statistic $z$ is calculated based on the standard normal distribution. 



Can you generate a random sample of size 1,000 and use the sample to test whether the probability that a randomly selected survey block is `NEAR OCEAN` is greater than 0.2?

H0: p = 0.2\
Ha: p > 0.2

Use **proportion.proportions_ztest** in `statsmodels.stats` to perform the one-sample z test. Because the alternative hypothesis is $p > 0.2$, this is an upper-tailed test.

In [8]:
n = 1000 # sample size
alpha = 0.05 # significance level
p_0=0.2 # hypothesized probability of success

# sample 1000 blocks and count those that are "NEAR OCEAN"
count = (CA_housing.sample(n).OceanProx=="NEAR OCEAN").sum()

z_stat, p_value = proportion.proportions_ztest(count=count,
                                               nobs=n,
                                               value=p_0,
                                               alternative='larger',#‘two-sided’, ‘smaller’, ‘larger’]
                                               prop_var=False) # If prop_var is false, the variance of the proportion estimate is calculated based on the sample proportion.
print('One-sample z test:')
print(f'The test statistic is {z_stat:.3f} and the p value is {p_value:.3f}')


if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

One-sample z test:
The test statistic is -3.641 and the p value is 1.000
Fail to reject the null hypothesis at the significance level 0.05 



## Two-sample z-test

Let $p_1$ and $p_2$ be the probabilities of success for two Bernoulli distributions, respectively. The two-sample $z$ test can be used to determine whether the two distributions have the same probability of success.

Let $n_1$ be the sample size of a random sample drawn from distribution 1. $\overline{p}_1$ is the sample proportion of success. Similarly, $n_2$ is the sample size of a sample drawn from distribution 2, and $\overline{p}_2$ is the sample proportion. The table below summarizes the two-sample z tests.

**Table 4: Hypotheses for two-sample z tests**

$$
\begin{array}{l l l l}
 & \text{lower-tailed test} & \text{upper-tailed test} & \text{two-tailed test} \\
\hline
\text{Null hypothesis } H_0: & p_1 \ge p_2 & p_1 \le p_2 & p_1 = p_2 \\
\text{Alternative hypothesis } H_a: & p_1 < p_2 & p_1 > p_2 & p_1 \neq p_2 \\
\text{p-value:} & P(x \le z) & P(x > z) & P(x \ge |z|) \\
\hline
\end{array}
$$

The test statistic is
\begin{equation}
z=\frac{\overline{p}_1-\overline{p}_2}{\sqrt{\overline{p}(1-\overline{p})(\frac{1}{n_1}+\frac{1}{n_2}})},\tag{9}
\end{equation}
where $\overline{p}$ is the pooled sample proportion:
\begin{equation}
\overline{p}=\frac{\overline{p}_1n_1+\overline{p}_2n_2}{n_1+n_2}. \tag{10}
\end{equation}

The pooled proportion combines the two samples into one estimate under the null hypothesis that the two population proportions are equal. The $p$-value in the table is calculated based on the standard normal distribution. 



Assume the first 5,000 rows of the dataset are from district 1, and the next 3,000 rows are from district 2.
Can you test whether the probability that a block randomly selected from district 1 is `NEAR OCEAN` is greater than that probability for district 2?

H0: p_1 $\leq$ p_2\
Ha: p_1 $>$ p_2

Use **proportion.proportions_ztest()** from `statsmodels.stats` to perform the two-sample z test. Because the alternative hypothesis is $p_1 > p_2$, this is an upper-tailed comparison.


In [9]:
# Test whether the "NEAR OCEAN" proportion in one district is larger than that in another district

Dist_1=CA_housing.iloc[0:5000,:]
Dist_2=CA_housing.iloc[500:8000,:]

n_1 = 500 # sample size for district 1
n_2 = 300 # sample size for district 2
alpha = 0.05 # significance level

c_1 = (Dist_1.sample(n_1).OceanProx=="NEAR OCEAN").sum() # count "NEAR OCEAN" blocks in group 1
c_2 = (Dist_2.sample(n_2).OceanProx=="NEAR OCEAN").sum() # count "NEAR OCEAN" blocks in group 2

z_stat, p_value = proportion.proportions_ztest(count=[c_1,c_2], 
                                               nobs=[n_1,n_2],  
                                               alternative='larger'
                                              ) # two-sided, larger, smaller
print('Two-sample z-test:')
print(f'Test statistic is {z_stat:.3f} and p value is {p_value:.3f},\n')


if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

Two-sample z-test:
Test statistic is 0.369 and p value is 0.356,

Fail to reject the null hypothesis at the significance level 0.05 



## One-way Chi-squared Goodness-of-fit Test

The one-way chi-square goodness-of-fit test examines whether categorical data follow a hypothesized frequency distribution. This test is useful when one categorical variable has several possible categories, and we want to compare observed counts with expected counts.

Consider $F$ as the distribution of a categorical variable that takes values from the set $\mathscr{A}=\{c_k|k=1,\dots, K\}$. Let $\{f_k|k=1,\dots,K\}$ be the hypothesized frequency distribution. The one-way $\chi^2$ test examines:

H0: the distribution that generates the data is $\{f_k|k=1,\dots,K\}$ <br>
Ha: the distribution that generates the data is different from $\{f_k|k=1,\dots,K\}$.

Given a random sample, $x_1, \dots, x_n$, drawn from the distribution $F$, we can estimate the frequency distribution, $\{\widehat{f}_k|k=1,\dots,K\}$:
\begin{equation}
\widehat{f}_k=\sum_{i=1}^K 1\{x_i=c_k\} \tag{10}
\end{equation}
for $i=1,\dots, k$. Then, we calculate the test statistic:
\begin{equation}
Q=\sum_{i=1}^K \frac{(\widehat{f}_k-f_k)^2}{f_k}. \tag{11}
\end{equation}

Each term compares an observed count with an expected count. Larger differences between observed and expected counts lead to a larger $Q$ statistic. The $p$-value is calculated based on the $\chi^2$ distribution with $K-1$ degrees of freedom. If the $p$-value is less than the significance level, $\alpha$, we can reject the null hypothesis.



A hypothesized distribution of survey blocks by `OceanProx` is [INLAND=0.15, NEAR BAY=0.1, <1H OCEAN=0.4, NEAR OCEAN=0.3, ISLAND=0.05]. Can you generate a random sample of 2,000 and use the sample to test whether the distribution that generated the sample is the same as the hypothesized distribution?


H0: f $=$ [0.15, 0.1, 0.4, 0.3, 0.05]\
Ha: f $\neq$ [0.15, 0.1, 0.4, 0.3, 0.05]

Use **scipy.stats.chisquare** to perform the one-way $\chi^2$ test. The key idea is to compare the observed category counts in the sample with the expected category counts under the hypothesized distribution.

In [10]:
n = 2000 # sample size
alpha = 0.05 # significance level

# what proportions do we expect to see?
expected_distribution = [0.15, 0.3, 0.1, 0.4, 0.05]


# what counts did we observe in our sample?
x=list(CA_housing['OceanProx'].sample(n))
observed_counts = [x.count('INLAND'), x.count('NEAR BAY'),x.count('<1H OCEAN'),x.count('NEAR OCEAN'),x.count('ISLAND')]
observed_counts


# counts based on expected proportions
expected_counts = list((np.array(expected_distribution) * n).astype(int))
expected_counts

# Get the test statistic and p-value. sp.stats.chisquare(observed_counts, expected_counts)
chi_stat, p_value = sp.stats.chisquare(observed_counts, expected_counts)

# report
print('The one-way chi-squared test:')
print(f'Test statistic is {chi_stat:.3f} and the p value is {p_value:.3f},\n')


if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

[606, 224, 907, 263, 0]

[np.int64(300), np.int64(600), np.int64(200), np.int64(800), np.int64(100)]

The one-way chi-squared test:
Test statistic is 3507.453 and the p value is 0.000,

Reject the null hypothesis at the significance level 0.05 



## Chi-squared Contingency Test

The $\chi^2$ contingency test is used to determine whether the distribution of a categorical variable is homogeneous across different populations. In this setting, "homogeneous" means that the category proportions are similar across the groups being compared.

For example, to determine whether the distribution of survey blocks by `OceanProx` is homogeneous across three districts, we selected one sample from each district and summarized the frequency distribution accordingly:

**Table 5: Distribution by District** 

| OceanProx    | District_1 | District_2 | District_3 |
|---------------|-------------|-------------|-------------|
| INLAND        | 164         | 44          | 283         |
| NEAR BAY      | 131         | 0           | 20          |
| <1H OCEAN     | 195         | 244         | 262         |
| NEAR OCEAN    | 10          | 12          | 34          |
| ISLAND        | 0           | 0           | 1           |


We can use the $\chi^2$ contingency test to examine whether the distribution is homogeneous across the three districts:

H0: the distributions are homogeneous<br>
Ha: the distributions are not homogeneous

The test statistic is:
\begin{equation}
Q=\sum_{k=1}^K\sum_{l=1}^L (O_{k,l}-E_{k,l})^2/E_{k,l}  \tag{12}
\end{equation}
where $O_{k,l}$ is the observed value of category $k$ from distribution $l$, $L$ is the number of distributions, and $K$ is the number of categories of the categorical variable. $E_{k,l}$ is the expected value of category $k$ from distribution $l$, computed as:
\begin{equation}
E_{k,l}=\frac{\sum_{k=1}^K O_{k,l}\sum_{l=1}^L O_{k,l}}{\sum_{l=1}^L\sum_{k=1}^K O_{k,l}}. \tag{13}
\end{equation}

The expected counts represent what we would expect to see if the category distribution were the same across all groups. The $p$-value is the probability that a random value drawn from the chi-square distribution with $(K-1)(L-1)$ degrees of freedom is greater than the test statistic $Q$. If the $p$-value at the test statistic is less than the level of significance $\alpha$, we can reject the null hypothesis and conclude that the distribution of the categorical variable is not homogeneous across different populations.


Assume the first 5,000 rows of `CA_housing` are in the first district, the next 3,000 rows are in the second district, and the following 6,000 rows are in the third district.

Let us draw a random sample of size 10% from each district. Using the sample data, can you test whether the distribution of `OceanProx` is homogeneous across the three districts?

H0: f_1 = f_2 = f_3\
Ha: $\exists\; f_i\neq f_j$, for $i, j \in\{1,2,3\}$

Use **scipy.stats.chi2_contingency** to perform the $\chi^2$ contingency test. If the test rejects $H_0$, the data suggest that at least one district has a different `OceanProx` distribution.

In [11]:
Dist_1=CA_housing.iloc[0:5000,:]
Dist_2=CA_housing.iloc[5000:8000,:]
Dist_3=CA_housing.iloc[8000:14000,:]

n_1, n_2, n_3 = (0.1*np.array([len(Dist_1), len(Dist_2), len(Dist_3)])).astype(int)
alpha = 0.05 # significance level


c_1 = Dist_1['OceanProx'].sample(n_1,random_state=0).value_counts()
c_2 = Dist_2['OceanProx'].sample(n_2,random_state=0).value_counts()
c_3 = Dist_3['OceanProx'].sample(n_3,random_state=0).value_counts()
my_table = pd.DataFrame({'District_1': c_1, 'District_2': c_2, 'District_3': c_3}).fillna(0)
my_table


chi_stat, p_value, degrees_of_freedom, expected = sp.stats.chi2_contingency(my_table.T.values)
print('Chi-squared contingincy test:')
print(f'Test statistic is {chi_stat:.3f} and the p value is {p_value:.3f}\n')

if p_value > alpha:
   print (f'Fail to reject the null hypothesis at the significance level {alpha}','\n')
else:
   print (f'Reject the null hypothesis at the significance level {alpha}','\n')

,District_1,District_2,District_3
OceanProx,,,
INLAND,164,44,283
NEAR BAY,131,0,20
<1H OCEAN,195,244,262
NEAR OCEAN,10,12,34
ISLAND,0,0,1


Chi-squared contingincy test:
Test statistic is 320.987 and the p value is 0.000

Reject the null hypothesis at the significance level 0.05 



Students who need to use sampling and statistical inference in depth should read textbooks carefully. 

Wasserman, Larry. All of statistics: a concise course in statistical inference. Springer Science & Business Media, 2013.

Ross, Sheldon M. Introduction to probability and statistics for engineers and scientists. Academic Press, 2020.
